# US Census MARTS — PM Quickstart

Load the committed April 2026 Advance Monthly Retail Trade Survey file, parse Table 1 (sales by NAICS), and produce a 'category growth at a glance' view a PM can drop in a deck.

**Dataset**: [`samples/census-marts-apr2026.xlsx`](../samples/census-marts-apr2026.xlsx) — public domain.

Source: https://www.census.gov/retail/

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

xls = pd.ExcelFile('../samples/census-marts-apr2026.xlsx')
print('Sheets:', xls.sheet_names)

## 1. Read Table 1 — Estimated Monthly Sales by NAICS

The headers span the top 10 rows. We skip them and label columns manually.

In [ ]:
raw = pd.read_excel(xls, sheet_name='Table 1.', header=None)
# Inspect rows 0-15 to locate the data
raw.head(15)

In [ ]:
# Headers occupy rows 0-9; data begins at row 10. Adjust if Census changes layout.
data = raw.iloc[10:].copy()
data.columns = [
    'NAICS',
    'KindOfBusiness',
    'TotalSales_4mo_2026',
    'PctChg_vs2025',
    'Apr_2026_NSA_adv',
    'Mar_2026_NSA_prelim',
    'Feb_2026_NSA_revised',
    'Apr_2025_NSA',
    'Mar_2025_NSA_revised',
    'Apr_2026_SA_adv',
    'Mar_2026_SA_prelim',
    'Feb_2026_SA_revised',
    'Apr_2025_SA',
    'Mar_2025_SA_revised',
]
data = data.dropna(subset=['KindOfBusiness']).reset_index(drop=True)
data.head(10)

## 2. Year-over-year growth by category

Using seasonally adjusted columns.

In [ ]:
data['YoY_pct'] = (data['Apr_2026_SA_adv'] - data['Apr_2025_SA']) / data['Apr_2025_SA'] * 100
data_with_growth = data.dropna(subset=['Apr_2026_SA_adv', 'Apr_2025_SA']).copy()
data_with_growth = data_with_growth.sort_values('YoY_pct', ascending=False)
data_with_growth[['NAICS', 'KindOfBusiness', 'Apr_2026_SA_adv', 'Apr_2025_SA', 'YoY_pct']].head(15)

In [ ]:
top10 = data_with_growth.head(10)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10['KindOfBusiness'].str.strip(), top10['YoY_pct'])
ax.invert_yaxis()
ax.set_xlabel('YoY % change')
ax.set_title('Fastest-growing retail categories — Apr 2026 vs Apr 2025 (SA)')
plt.tight_layout()
plt.show()

## What to try next
- Download a full history (all MARTS releases 2010-present) → time series.
- Compare your business unit's YoY to the NAICS line you compete in (NAICS 4541 = electronic shopping / mail-order).
- Read [`scorecards/04-forecasting-accuracy.md`](../scorecards/04-forecasting-accuracy.md) — Census is the macro anchor for forecast reconciliation.